# Laser Etching — Q-Factor Analysis
Consolidated analysis of 423 z-map images (Silicon Wafer).
Covers distributions, process parameter effects, A/R component analysis, and spatial uniformity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

plt.style.use('seaborn-v0_8-darkgrid')
CSV_PATH         = 'qfactors_1.0um.csv'
SEGMENT_WIDTH_UM = 1.0
FEATURES         = ['scan_speed', 'n_pass', 'range_um', 'power']


## 1. Load

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df['q_factor'].notna() & df['q_factor'].between(0, 1)]

# Per-image aggregation
meta = (df.drop_duplicates('filename')
          [['filename','Power(W)','Scanning speed(mm/s)','Pass','Range(micron)']]
          .rename(columns={'Power(W)':'power','Scanning speed(mm/s)':'scan_speed',
                            'Pass':'n_pass','Range(micron)':'range_um'}))

agg = df.groupby('filename').agg(
    q_median  = ('q_factor', 'median'),
    q_std     = ('q_factor', 'std'),
    A_median  = ('A',        'median'),
    R_median  = ('R',        'median'),
    A_sum     = ('A',        'sum'),
).reset_index()
agg['volume_um3'] = agg['A_sum'] * SEGMENT_WIDTH_UM / 1000

img = meta.merge(agg, on='filename')

# Relative position for spatial analysis
pos_min        = df.groupby('filename')['position_um'].transform('min')
pos_max        = df.groupby('filename')['position_um'].transform('max')
pos_rng        = pos_max - pos_min
df             = df.copy()
df['rel_pos']  = np.where(pos_rng > 0, (df['position_um'] - pos_min) / pos_rng, 0.5)
N_BINS         = 25
df['pos_bin']  = (df['rel_pos'] * N_BINS).astype(int).clip(0, N_BINS - 1)
bin_x          = np.arange(N_BINS) / N_BINS + 0.5 / N_BINS

print(f'Valid slices : {len(df):,}')
print(f'Images       : {len(img)}')


## 2. Dataset Overview

In [ ]:
rows = [
    {'quantity':'Q-factor',         'level':'per-image','unit':'—',      'n':len(img),
     'mean':img['q_median'].mean(), 'std':img['q_median'].std(),
     'median':img['q_median'].median(),'min':img['q_median'].min(),'max':img['q_median'].max(),
     'skew':img['q_median'].skew()},
    {'quantity':'Ablated area A',     'level':'per-slice','unit':'nm·µm','n':len(df),
     'mean':df['A'].mean(),          'std':df['A'].std(),
     'median':df['A'].median(),      'min':df['A'].min(),'max':df['A'].max(),
     'skew':df['A'].skew()},
    {'quantity':'Redeposited area R', 'level':'per-slice','unit':'nm·µm','n':len(df),
     'mean':df['R'].mean(),          'std':df['R'].std(),
     'median':df['R'].median(),      'min':df['R'].min(),'max':df['R'].max(),
     'skew':df['R'].skew()},
    {'quantity':'Removed volume',     'level':'per-image','unit':'µm³',    'n':len(img),
     'mean':img['volume_um3'].mean(),'std':img['volume_um3'].std(),
     'median':img['volume_um3'].median(),'min':img['volume_um3'].min(),'max':img['volume_um3'].max(),
     'skew':img['volume_um3'].skew()},
]
pd.set_option('display.float_format', '{:.3f}'.format)
print(pd.DataFrame(rows).set_index('quantity').to_string())


## 3. Distributions

In [ ]:
def dist_ax(ax, vals, xlabel, color):
    ax.hist(vals, bins=50, density=True, color=color, alpha=0.55)
    kde = gaussian_kde(vals, bw_method=0.2)
    xk  = np.linspace(vals.min(), vals.max(), 400)
    ax.plot(xk, kde(xk), color=color, linewidth=2)
    ax.axvline(vals.mean(),      color='red',    linestyle='--', linewidth=1.4,
               label=f'Mean={vals.mean():.3f}')
    ax.axvline(np.median(vals),  color='orange', linestyle='--', linewidth=1.4,
               label=f'Median={np.median(vals):.3f}')
    ax.text(0.97, 0.97,
            f'n={len(vals):,}\nstd={vals.std():.3f}\nskew={pd.Series(vals).skew():.2f}',
            transform=ax.transAxes, va='top', ha='right', fontsize=8,
            bbox=dict(boxstyle='round', alpha=0.15))
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
dist_ax(axes[0,0], img['q_median'].values,   'Q-factor  (per-image median)',      'steelblue')
dist_ax(axes[0,1], df['A'].values,            'Ablated area A  (nm·µm, per-slice)', 'darkorange')
dist_ax(axes[1,0], df['R'].values,            'Redeposited area R  (nm·µm, per-slice)', 'seagreen')
dist_ax(axes[1,1], img['volume_um3'].values,  'Removed volume  (µm³, per-image)',  'mediumpurple')
axes[0,0].set_title('Q-factor')
axes[0,1].set_title('Ablated Area (A)')
axes[1,0].set_title('Redeposited Area (R)')
axes[1,1].set_title('Removed Volume')
plt.suptitle('Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Process Parameter Effects
Median ± IQR across the 46 scan-speed levels and 9 n_pass levels.

In [ ]:
def line_ax(ax, grp_col, val_col, color, xlabel, ylabel):
    grp = img.groupby(grp_col)[val_col]
    med = grp.median()
    q1  = grp.quantile(0.25)
    q3  = grp.quantile(0.75)
    ax.plot(med.index, med.values, color=color, linewidth=2, marker='o', markersize=3)
    ax.fill_between(med.index, q1.values, q3.values, alpha=0.25, color=color, label='IQR')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
line_ax(axes[0,0], 'scan_speed', 'q_median',   'steelblue',    'Scan Speed (mm/s)',  'Q-factor')
line_ax(axes[0,1], 'n_pass',     'q_median',   'steelblue',    'Number of Passes',   'Q-factor')
line_ax(axes[1,0], 'scan_speed', 'volume_um3', 'mediumpurple', 'Scan Speed (mm/s)',  'Removed Volume (µm³)')
line_ax(axes[1,1], 'n_pass',     'volume_um3', 'mediumpurple', 'Number of Passes',   'Removed Volume (µm³)')
axes[0,0].set_title('Q vs Scan Speed')
axes[0,1].set_title('Q vs Number of Passes')
axes[1,0].set_title('Volume vs Scan Speed')
axes[1,1].set_title('Volume vs Number of Passes')
plt.suptitle('Process Parameter Effects  (median ± IQR)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 5. A/R Component Analysis
Q = 1 − R/A. Modelling A and R separately reveals where predictability actually lies.

In [ ]:
# Best single-param R^2 (poly deg 1-3) for Q, A, R
def best_r2(x, y):
    best = -np.inf
    for deg in [1, 2, 3]:
        try:
            yp = np.polyval(np.polyfit(x, y, deg), x)
            best = max(best, r2_score(y, yp))
        except Exception:
            pass
    return best

r2_Q = {f: best_r2(img[f].values, img['q_median'].values) for f in FEATURES}
r2_A = {f: best_r2(img[f].values, img['A_median'].values) for f in FEATURES}
r2_R = {f: best_r2(img[f].values, img['R_median'].values) for f in FEATURES}

# Multi-param Poly-2 + Ridge CV R^2
def poly_ridge_cv(X, y):
    pipe = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), Ridge(alpha=1.0))
    return cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()

X    = img[FEATURES].values
cv_Q = poly_ridge_cv(X, img['q_median'].values)
cv_A = poly_ridge_cv(X, img['A_median'].values)
cv_R = poly_ridge_cv(X, img['R_median'].values)

print(f'Multi-param Poly-2+Ridge CV R²:  Q={cv_Q:.3f}  A={cv_A:.3f}  R={cv_R:.3f}')


In [ ]:
x_pos  = np.arange(len(FEATURES))
w      = 0.25
colors = ['steelblue', 'darkorange', 'seagreen']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Single-param grouped bar
ax = axes[0]
ax.bar(x_pos - w, [r2_Q[f] for f in FEATURES], w, label='Q', color=colors[0])
ax.bar(x_pos,     [r2_A[f] for f in FEATURES], w, label='A', color=colors[1])
ax.bar(x_pos + w, [r2_R[f] for f in FEATURES], w, label='R', color=colors[2])
ax.set_xticks(x_pos)
ax.set_xticklabels(FEATURES)
ax.set_ylabel('Best R² (train)')
ax.set_title('Single-param: Q vs A vs R')
ax.legend()

# Multi-param CV R^2
ax = axes[1]
bars = ax.bar(['Q', 'A', 'R'], [cv_Q, cv_A, cv_R], color=colors, width=0.4)
for bar, val in zip(bars, [cv_Q, cv_A, cv_R]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom')
ax.set_ylabel('R² (5-fold CV)')
ax.set_title('Multi-param Poly-2+Ridge  CV R²')
ax.set_ylim(0, max(cv_Q, cv_A, cv_R) * 1.35)

plt.suptitle('A/R Component Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 6. Spatial Uniformity
**Intra-image Q std**: how uniform is quality along a single trench?
**Q profile**: does quality change from the start to the end of a scan?

In [ ]:
colors_3 = ['steelblue', 'darkorange', 'seagreen']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Q std distribution
ax = axes[0]
vals = img['q_std'].values
ax.hist(vals, bins=40, density=True, color='coral', alpha=0.55)
kde = gaussian_kde(vals, bw_method=0.3)
xk  = np.linspace(vals.min(), vals.max(), 300)
ax.plot(xk, kde(xk), color='coral', linewidth=2)
ax.axvline(vals.mean(),     color='red',    linestyle='--', linewidth=1.4,
           label=f'Mean={vals.mean():.3f}')
ax.axvline(np.median(vals), color='orange', linestyle='--', linewidth=1.4,
           label=f'Median={np.median(vals):.3f}')
ax.set_xlabel('Q std  (within image)')
ax.set_ylabel('Density')
ax.set_title('Q Uniformity Distribution')
ax.legend(fontsize=8)

# Q std vs scan_speed
ax = axes[1]
grp = img.groupby('scan_speed')['q_std']
ax.plot(grp.median().index, grp.median().values, color='coral', linewidth=2, marker='o', markersize=3)
ax.fill_between(grp.median().index, grp.quantile(0.25).values, grp.quantile(0.75).values,
                alpha=0.25, color='coral', label='IQR')
ax.set_xlabel('Scan Speed (mm/s)')
ax.set_ylabel('Q std')
ax.set_title('Q Uniformity vs Scan Speed')
ax.legend(fontsize=8)

# Q std vs n_pass
ax = axes[2]
grp2 = img.groupby('n_pass')['q_std']
ax.plot(grp2.median().index, grp2.median().values, color='coral', linewidth=2, marker='o', markersize=5)
ax.fill_between(grp2.median().index, grp2.quantile(0.25).values, grp2.quantile(0.75).values,
                alpha=0.25, color='coral', label='IQR')
ax.set_xlabel('Number of Passes')
ax.set_ylabel('Q std')
ax.set_title('Q Uniformity vs Number of Passes')
ax.legend(fontsize=8)

plt.suptitle('Intra-image Q Uniformity', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
df_sp = df.merge(img[['filename','scan_speed','n_pass']], on='filename')

speed_labels = ['Low (<700)',  'Mid (700-1800)', 'High (>1800)']
pass_labels  = ['Few (1-5)',   'Medium (10-20)', 'Many (25+)']
df_sp['speed_grp'] = pd.cut(df_sp['scan_speed'], bins=[0,700,1800,9999], labels=speed_labels)
df_sp['pass_grp']  = pd.cut(df_sp['n_pass'],     bins=[0,5,20,999],      labels=pass_labels)

# Overall profile
prof_all = df_sp.groupby('pos_bin')['q_factor'].median()
prof_q1  = df_sp.groupby('pos_bin')['q_factor'].quantile(0.25)
prof_q3  = df_sp.groupby('pos_bin')['q_factor'].quantile(0.75)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall
ax = axes[0]
ax.plot(bin_x, prof_all.values, color='steelblue', linewidth=2)
ax.fill_between(bin_x, prof_q1.values, prof_q3.values, alpha=0.25, color='steelblue', label='IQR')
ax.set_xlabel('Relative position  (0=start, 1=end)')
ax.set_ylabel('Q-factor')
ax.set_title('Overall Q Profile')
ax.legend(fontsize=8)

# By scan_speed
ax = axes[1]
for label, color in zip(speed_labels, colors_3):
    sub  = df_sp[df_sp['speed_grp'] == label]
    prof = sub.groupby('pos_bin')['q_factor'].median()
    ax.plot(bin_x[:len(prof)], prof.values, color=color, linewidth=2,
            label=f'{label} (n={sub["filename"].nunique()})')
ax.set_xlabel('Relative position')
ax.set_ylabel('Q-factor')
ax.set_title('Q Profile by Scan Speed')
ax.legend(fontsize=8)

# By n_pass
ax = axes[2]
for label, color in zip(pass_labels, colors_3):
    sub  = df_sp[df_sp['pass_grp'] == label]
    prof = sub.groupby('pos_bin')['q_factor'].median()
    ax.plot(bin_x[:len(prof)], prof.values, color=color, linewidth=2,
            label=f'{label} (n={sub["filename"].nunique()})')
ax.set_xlabel('Relative position')
ax.set_ylabel('Q-factor')
ax.set_title('Q Profile by Number of Passes')
ax.legend(fontsize=8)

plt.suptitle('Q-factor Profile Along Trench', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 7. Export
Run the full notebook top-to-bottom before executing this cell.

In [ ]:
from pathlib import Path

OUT = Path('results')
OUT.mkdir(exist_ok=True)

def dist_ax(ax, vals, xlabel, color):
    ax.hist(vals, bins=50, density=True, color=color, alpha=0.55)
    kde = gaussian_kde(vals, bw_method=0.2)
    xk  = np.linspace(vals.min(), vals.max(), 400)
    ax.plot(xk, kde(xk), color=color, linewidth=2)
    ax.axvline(vals.mean(),     color='red',    linestyle='--', linewidth=1.4,
               label=f'Mean={vals.mean():.3f}')
    ax.axvline(np.median(vals), color='orange', linestyle='--', linewidth=1.4,
               label=f'Median={np.median(vals):.3f}')
    ax.text(0.97, 0.97,
            f'n={len(vals):,}\nstd={vals.std():.3f}\nskew={pd.Series(vals).skew():.2f}',
            transform=ax.transAxes, va='top', ha='right', fontsize=8,
            bbox=dict(boxstyle='round', alpha=0.15))
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

def line_ax(ax, grp_col, val_col, color, xlabel, ylabel):
    grp = img.groupby(grp_col)[val_col]
    med = grp.median()
    q1  = grp.quantile(0.25)
    q3  = grp.quantile(0.75)
    ax.plot(med.index, med.values, color=color, linewidth=2, marker='o', markersize=3)
    ax.fill_between(med.index, q1.values, q3.values, alpha=0.25, color=color, label='IQR')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()

colors_3 = ['steelblue', 'darkorange', 'seagreen']

# 1. Distributions
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
dist_ax(axes[0,0], img['q_median'].values,  'Q-factor (per-image median)',       'steelblue')
dist_ax(axes[0,1], df['A'].values,           'Ablated area A (nm·µm, per-slice)', 'darkorange')
dist_ax(axes[1,0], df['R'].values,           'Redeposited area R (nm·µm, per-slice)', 'seagreen')
dist_ax(axes[1,1], img['volume_um3'].values, 'Removed volume (µm³, per-image)',   'mediumpurple')
for ax, t in zip(axes.flat, ['Q-factor','Ablated Area (A)','Redeposited Area (R)','Removed Volume']):
    ax.set_title(t)
plt.suptitle('Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: distributions.png')

# 2. Process parameter effects
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
line_ax(axes[0,0],'scan_speed','q_median',  'steelblue',   'Scan Speed (mm/s)','Q-factor')
line_ax(axes[0,1],'n_pass',    'q_median',  'steelblue',   'Number of Passes', 'Q-factor')
line_ax(axes[1,0],'scan_speed','volume_um3','mediumpurple','Scan Speed (mm/s)','Removed Volume (µm³)')
line_ax(axes[1,1],'n_pass',    'volume_um3','mediumpurple','Number of Passes', 'Removed Volume (µm³)')
for ax, t in zip(axes.flat,['Q vs Scan Speed','Q vs N Passes','Volume vs Scan Speed','Volume vs N Passes']):
    ax.set_title(t)
plt.suptitle('Process Parameter Effects (median ± IQR)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'param_effects.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: param_effects.png')

# 3. A/R component analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x_pos = np.arange(len(FEATURES))
w     = 0.25
axes[0].bar(x_pos-w, [r2_Q[f] for f in FEATURES], w, label='Q', color='steelblue')
axes[0].bar(x_pos,   [r2_A[f] for f in FEATURES], w, label='A', color='darkorange')
axes[0].bar(x_pos+w, [r2_R[f] for f in FEATURES], w, label='R', color='seagreen')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(FEATURES)
axes[0].set_ylabel('Best R² (train)')
axes[0].set_title('Single-param: Q vs A vs R')
axes[0].legend()
bars = axes[1].bar(['Q','A','R'],[cv_Q,cv_A,cv_R],color=['steelblue','darkorange','seagreen'],width=0.4)
for bar, val in zip(bars,[cv_Q,cv_A,cv_R]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}',ha='center',va='bottom')
axes[1].set_ylabel('R² (5-fold CV)')
axes[1].set_title('Multi-param Poly-2+Ridge  CV R²')
axes[1].set_ylim(0, max(cv_Q,cv_A,cv_R)*1.35)
plt.suptitle('A/R Component Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'ar_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: ar_analysis.png')

# 4. Spatial uniformity
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
# Row 1: uniformity
vals = img['q_std'].values
axes[0,0].hist(vals, bins=40, density=True, color='coral', alpha=0.55)
kde = gaussian_kde(vals, bw_method=0.3)
xk = np.linspace(vals.min(), vals.max(), 300)
axes[0,0].plot(xk, kde(xk), color='coral', linewidth=2)
axes[0,0].axvline(vals.mean(), color='red', linestyle='--', linewidth=1.4, label=f'Mean={vals.mean():.3f}')
axes[0,0].axvline(np.median(vals), color='orange', linestyle='--', linewidth=1.4, label=f'Median={np.median(vals):.3f}')
axes[0,0].set_xlabel('Q std'); axes[0,0].set_ylabel('Density')
axes[0,0].set_title('Q Uniformity Distribution'); axes[0,0].legend(fontsize=8)
for ax, grp_col, xlabel in [(axes[0,1],'scan_speed','Scan Speed (mm/s)'),
                             (axes[0,2],'n_pass','Number of Passes')]:
    grp = img.groupby(grp_col)['q_std']
    ax.plot(grp.median().index, grp.median().values, color='coral', linewidth=2, marker='o', markersize=3)
    ax.fill_between(grp.median().index, grp.quantile(0.25).values, grp.quantile(0.75).values,
                    alpha=0.25, color='coral', label='IQR')
    ax.set_xlabel(xlabel); ax.set_ylabel('Q std')
    ax.set_title(f'Q Uniformity vs {xlabel}'); ax.legend(fontsize=8)
# Row 2: spatial profiles
prof_all = df_sp.groupby('pos_bin')['q_factor'].median()
prof_q1  = df_sp.groupby('pos_bin')['q_factor'].quantile(0.25)
prof_q3  = df_sp.groupby('pos_bin')['q_factor'].quantile(0.75)
axes[1,0].plot(bin_x, prof_all.values, color='steelblue', linewidth=2)
axes[1,0].fill_between(bin_x, prof_q1.values, prof_q3.values, alpha=0.25, color='steelblue', label='IQR')
axes[1,0].set_xlabel('Relative position'); axes[1,0].set_ylabel('Q-factor')
axes[1,0].set_title('Overall Q Profile'); axes[1,0].legend(fontsize=8)
for ax, grp_col, labels in [(axes[1,1],'speed_grp',speed_labels),
                             (axes[1,2],'pass_grp', pass_labels)]:
    for label, color in zip(labels, colors_3):
        sub  = df_sp[df_sp[grp_col] == label]
        prof = sub.groupby('pos_bin')['q_factor'].median()
        ax.plot(bin_x[:len(prof)], prof.values, color=color, linewidth=2,
                label=f'{label} (n={sub["filename"].nunique()})')
    ax.set_xlabel('Relative position'); ax.set_ylabel('Q-factor')
    ax.set_title('Q Profile by ' + ('Scan Speed' if grp_col=='speed_grp' else 'N Passes'))
    ax.legend(fontsize=7)
plt.suptitle('Spatial Uniformity', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / 'spatial_uniformity.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: spatial_uniformity.png')

# 5. Summary CSV
rows = []
for f in FEATURES:
    rows.append({'parameter':f,'r2_Q':round(r2_Q[f],3),'r2_A':round(r2_A[f],3),'r2_R':round(r2_R[f],3)})
rows.append({'parameter':'all (multi-param CV)','r2_Q':round(cv_Q,3),'r2_A':round(cv_A,3),'r2_R':round(cv_R,3)})
pd.DataFrame(rows).to_csv(OUT / 'analysis_summary.csv', index=False)
print('Saved: analysis_summary.csv')
print()
print('All files saved to:', OUT.resolve())
